# Create Louisiana Board of Regents (BoRSF RCS) Awards

Creates Louisiana Board of Regents awards from the Board of Regents Support Fund (BoRSF)
**Research Competitiveness Subprogram (RCS)** final panel reports that the Board publishes at
rsi.laregents.edu (WordPress media, e.g. `.../2025/04/RCS_2025_FINAL-1.pdf`). Each report lists
the proposals "highly recommended for funding" (Table I / Appendix A, with the recommended amount
per year) and one block per funded proposal with proposal number, title, institution and PI.
**101 awards, contracts 2023-2026, USD 14.7M** (local run 2026-10-01).

**Scope / filter:** state BoRSF money only. RCS competitions FY 2022-23 .. FY 2025-26 (contracts
starting 2023-2026). Not included: the FY 2019-20 .. 2021-22 RCS reports (their Priority-One lists
include proposals that were never contracted - e.g. 5 of the 36 FY 2019-20 recommendations - so
they are not the awarded set); the other BoRSF subprograms (ITRS, PoCP, ATLAS, Enhancement,
graduate fellowships, endowed chairs/professorships, SREB), which are separate reports in other
layouts; and EPSCoR/LAMDA awards (NSF/NASA money).

**Prerequisites:**
- Run `scripts/local/la_borsf_rcs_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/la_borsf_rcs/la_borsf_rcs_projects.parquet` (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** papers cite the BoRSF contract number
`LEQSF(<first FY>-<last FY>)-RD-A-<NN>` (e.g. `LEQSF(2024-27)-RD-A-28`; ~285 RD-A citation stubs
under F4320309392, checked 2026-10-01). The reports print only the proposal number (`097A-24`), but
NN is the proposal's rank in ascending proposal-number order among the competition's funded
proposals, and the year span is the award's duration (1-year `(2024-25)`, 2-year `(2023-25)`,
3-year `(2024-27)`; one sequence across durations). The derived numbers were checked against the
works citing each 2023-2026 RD-A contract: the predicted PI is an author for 30 of 32 cited
contracts (the misses look like citation typos). The proposal number is kept in the raw table.
Zero-padded `RD-A-05` is the dominant cited spelling and is what we ship.

**Amounts / dates:** USD, the panel's recommended amount summed over the award years. BoRSF
contracts run June 1 - May 31, so `start_date` = June 1 of the first contract year and `end_date` =
May 31 of the last.

**lead_investigator:** the PI named in the report, split in Python (runbook §2.4.1), with the
institution as affiliation.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320309392` (Louisiana Board of Regents, US; ror 00jv89z46; doi 10.13039/100006952)

**Priority:** `547` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.la_borsf_rcs_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/la_borsf_rcs/la_borsf_rcs_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_rows, MIN(contract_number), MAX(contract_number)
FROM openalex.awards.la_borsf_rcs_raw;

## Step 1.5: Inspect raw data

In [ ]:
%sql
DESCRIBE openalex.awards.la_borsf_rcs_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.la_borsf_rcs_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320309392 is a Crossref-registered funder, so it MUST resolve to exactly 1 row in
`openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320309392;

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'F4320309392 must have exactly one row in openalex.funders.funders') AS funder_ok
FROM openalex.funders.funders
WHERE funder_id = 4320309392;

## Step 2: Create Louisiana BoRSF RCS Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.la_borsf_rcs_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320309392  -- Louisiana Board of Regents
),

lead AS (
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        contract_number,
        named_struct(
            'given_name', NULLIF(TRIM(lead_given_name), ''),
            'family_name', NULLIF(TRIM(lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(institution), ''),
                'country', 'US',
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) AS lead_investigator
    FROM openalex.awards.la_borsf_rcs_raw
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.contract_number))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.contract_number) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN 'USD' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'grant' as funding_type,
    'Board of Regents Support Fund - Research Competitiveness Subprogram (RCS)' as funder_scheme,
    'la_borsf_rcs' as provenance,
    TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
    TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
    YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')) as start_year,
    YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,
    l.lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    array(l.lead_investigator) as investigators,
    g.report_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.contract_number))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.la_borsf_rcs_raw g
JOIN lead l ON l.contract_number = g.contract_number
CROSS JOIN src_funder f
WHERE g.contract_number IS NOT NULL AND TRIM(g.contract_number) != '';

In [ ]:
%sql
-- Shape check (§2.1.1): every funder_award_id is a BoRSF RCS contract number; ids unique.
SELECT
    assert_true(COUNT_IF(NOT funder_award_id RLIKE '^LEQSF\\(20[0-9]{2}-[0-9]{2}\\)-RD-A-[0-9]{2}$') = 0,
                'funder_award_id is not an LEQSF(YYYY-YY)-RD-A-NN contract number') AS shape_ok,
    assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') AS unique_ok
FROM openalex.awards.la_borsf_rcs_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'la_borsf_rcs' AND priority = 547;

-- Insert into openalex_awards_raw with priority.
-- Priority 547: direct-from-funder ingest of the Louisiana BoRSF RCS panel reports.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    547 as priority
FROM openalex.awards.la_borsf_rcs_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.la_borsf_rcs_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, amount, currency, start_date, end_date,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.la_borsf_rcs_awards LIMIT 20;

In [ ]:
%sql
SELECT start_year, end_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_usd
FROM openalex.awards.la_borsf_rcs_awards
GROUP BY start_year, end_year ORDER BY start_year, end_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.la_borsf_rcs_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.la_borsf_rcs_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    collect_set(currency) as currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name
FROM openalex.awards.la_borsf_rcs_awards;

In [ ]:
%sql
-- Existing citation shells for F4320309392 that these awards upgrade.
SELECT c.provenance, COUNT(*) as shells, COUNT(t.id) as matched_by_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.la_borsf_rcs_awards t ON t.id = c.id
WHERE c.funder_id = 4320309392 AND c.priority < 3
GROUP BY c.provenance;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'la_borsf_rcs'
GROUP BY provenance, priority;